# GDELT Article Body Scraper

This notebook iterates over the URLs in the cleaned GDELT dataset and attempts to 
extract the body text of each article using HTTP requests and BeautifulSoup HTML 
parsing. For each article, irrelevant HTML elements (scripts, navigation, footers) 
are removed and the first three substantive paragraphs are extracted. This body text 
will later be used alongside article titles to compare the quality of sentiment 
signals extracted by FinBERT from headlines only versus headlines plus body text — 
an experiment designed to quantify headline bias in financial sentiment analysis.

**Pipeline:**
1. Scrape all articles → save raw immediately to `01_data/raw/news/gdelt_wti_with_body_raw.csv`
2. Clean characters and filter boilerplate → save to `01_data/processed/gdelt_wti_with_body.csv`

### Necessary imports

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import html

### Read processed news

In [2]:
df_clean = pd.read_csv("../01_data/processed/gdelt_wti_clean.csv", parse_dates=['datetime'])
print(f"Articles loaded: {len(df_clean)}")

Articles loaded: 3290


### Scraping function

Discards irrelevant HTML elements such as `script`, `style`, `nav`, `footer`, `header`, `aside` and extracts the first 3 substantive paragraphs (>50 characters) of each article.

In [3]:
def scrape_article(url, timeout=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove irrelevant HTML elements
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()

        # Extract first 3 substantive paragraphs
        paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 50]
        body = ' '.join(paragraphs[:3])
        return body

    except Exception as e:
        return f"ERROR: {str(e)[:100]}"

### Test scraping on 5 sample articles

In [14]:
test_urls = df_clean.sample(5, random_state=42)[['title', 'url', 'domain']].reset_index(drop=True)

for _, row in test_urls.iterrows():
    print(f"\nDomain: {row['domain']}")
    print(f"Title: {row['title'][:80]}")
    body = scrape_article(row['url'])
    print(f"Body ({len(body)} chars): {body[:200]}")
    print("-" * 60)


Domain: lelezard.com
Title: NOG Announces Third Quarter 2024 Results , Achieves Record Oil Production
Body (1305 chars): Northern Oil and Gas, Inc. (NYSE: NOG) ("NOG" or "Company") today announced the Company's third quarter results. "During the third quarter we generated record oil volumes and free cash flow despite li
------------------------------------------------------------

Domain: fxstreet.com
Title: Canadian Dollar steadies as US shutdown freezes economic data
Body (351 chars): • CAD fell against the USD as markets digest ongoing US government shutdown and suspended economic data releases. • BLS announced Friday'sNFPjobs report is suspended until federal operations resume; U
------------------------------------------------------------

Domain: londonlovesbusiness.com
Title: Oil opened higher on sanctions and OPEC+ plans , uncertainty lingers  - London B
Body (0 chars): 
------------------------------------------------------------

Domain: banklesstimes.com
Title: Here Why the

### Scrape all articles

Iterates over all articles and extracts body text. Progress is printed every 50 articles.

**Warning:** This cell takes 30-60 minutes to complete. Raw results are saved immediately after the loop finishes — do not interrupt before saving.

In [ ]:
bodies = []
total = len(df_clean)

for i, row in df_clean.iterrows():
    body = scrape_article(row['url'])
    bodies.append(body)

    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{total} — {row['domain']}")

    time.sleep(0.5)

df_clean['body'] = bodies

# Save raw immediately — before any filtering or cleaning
filename = "gdelt_wti_with_body_raw.csv"
df_clean.to_csv(f"../01_data/raw/news/{filename}", index=False)
print(f"\nRaw body data saved to 01_data/raw/news/{filename}")

# Summary
successful = df_clean[~df_clean['body'].str.startswith('ERROR') & (df_clean['body'].str.len() > 100)]
print(f"Total articles: {total}")
print(f"With body extracted: {len(successful)} ({len(successful)/total*100:.1f}%)")
print(f"Failed or empty: {total - len(successful)}")

### Diagnostic — check for dirty characters

Reads from the raw CSV to check for HTML entities, newlines, and control characters that could corrupt downstream processing.

In [4]:
df_raw = pd.read_csv("../01_data/raw/news/gdelt_wti_with_body_raw.csv")

issues = []
for i, row in df_raw.iterrows():
    body = str(row['body'])
    if '&amp;' in body or '&nbsp;' in body or '&#' in body:
        issues.append((i, 'HTML entities', body[:100]))
    if '\n' in body or '\r' in body:
        issues.append((i, 'Newlines', body[:100]))
    if re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', body):
        issues.append((i, 'Control characters', body[:100]))

print(f"Articles with issues: {len(issues)}")
for idx, issue_type, sample in issues[:10]:
    print(f"\n[{issue_type}] Row {idx}: {sample}")

Articles with issues: 0


### Clean and filter

Reads from the raw CSV and applies two transformations:

- **Character cleaning:** Unescapes HTML entities, removes control characters, normalizes whitespace.
- **Boilerplate filtering:** Allow-list approach where the news are inspected for specific words that potentially are from a legit news instead of boilerplate, errors, empty strings, etc.

In [5]:
"""Character cleaning"""
df_raw = pd.read_csv("../01_data/raw/news/gdelt_wti_with_body_raw.csv")

# --- Character cleaning ---
def clean_body(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['body'] = df_raw['body'].apply(clean_body)
print("Character cleaning done.")

Character cleaning done.


In [8]:
"""Boilerplate allow-listing"""

# Keywords that should appear in legitimate WTI-related financial content
ALLOW_KEYWORDS = [
    'oil', 'crude', 'wti', 'brent', 'opec', 'barrel', 'petroleum',
    'energy', 'price', 'market', 'supply', 'demand', 'production',
    'refinery', 'inventory', 'futures', 'trading', 'commodity',
    'inflation', 'fed', 'dollar', 'interest rate', 'gdp'
]

press_release_phrases = [
    "newsfile corp",
    "tsxv:",
    "otcqx:",
    "tsx:",
    "nyse:",
    "non-ifrs",
    "international financial reporting standards",
    "this press release",
    "forward-looking statements",
    "is pleased to announce",
    "is pleased to provide",
]

def is_valid_body(text):
    if not isinstance(text, str):
        return False
    if len(text) < 400:
        return False
    if text.startswith('ERROR'):
        return False
    
    text_lower = text.lower()
    
    # Must contain at least one energy/financial keyword
    has_keyword = any(kw in text_lower for kw in ALLOW_KEYWORDS)
    if not has_keyword:
        return False
    if any(phrase in text_lower for phrase in press_release_phrases):
        return False
    
    # Reject if more than 30% of text is capitalized words (navigation/boilerplate pattern)
    words = text.split()
    if len(words) > 10:
        cap_ratio = sum(1 for w in words if w.isupper() and len(w) > 2) / len(words)
        if cap_ratio > 0.3:
            return False
    
    return True

df_raw['body_valid'] = df_raw['body'].apply(is_valid_body)
print(f"Valid bodies: {df_raw['body_valid'].sum()}")
print(f"Discarded: {(~df_raw['body_valid']).sum()}")
print(df_raw['body'].str.len().describe().round(0))

# Save processed
filename = "gdelt_wti_with_body.csv"
df_raw.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"\nSaved to 01_data/processed/{filename}")

Valid bodies: 1900
Discarded: 1390
count     3290.0
mean       813.0
std       1767.0
min          0.0
25%        204.0
50%        536.0
75%        882.0
max      42495.0
Name: body, dtype: float64

Saved to 01_data/processed/gdelt_wti_with_body.csv
